In [ ]:
import pandas as pd
import numpy as np
import os
import time
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [2]:
df = pd.read_csv('./corr_output/전체_C_D_상관계수_통합.csv')

In [6]:
corr_df = df[df['correlation'].abs() >= 0.3]
corr_df

,feature,correlation,source
42,이용금액_R3M_신용체크,-0.306900,1_회원정보
47,_1순위카드이용금액,-0.318514,1_회원정보
138,이용금액_일시불_B0M,-0.303103,3_승인매출정보
166,이용금액_일시불_R12M,-0.326401,3_승인매출정보
492,정상청구원금_B0M,-0.423459,3_승인매출정보
494,정상입금원금_B0M,-0.314765,3_승인매출정보
496,정상청구원금_B2M,-0.416269,3_승인매출정보
498,정상입금원금_B2M,-0.305066,3_승인매출정보
500,정상청구원금_B5M,-0.438868,3_승인매출정보
502,정상입금원금_B5M,-0.309526,3_승인매출정보


In [7]:
selected_columns = corr_df['feature'].tolist()

# 데이터 경로 목록
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

# 필요한 컬럼들을 담을 리스트
df_list = []

for path in paths:
    df = pd.read_parquet(path)
    # 현재 파일에서 필요한 컬럼만 추출
    common_cols = list(set(df.columns) & set(selected_columns))
    if common_cols:
        df_list.append(df[common_cols])

# 컬럼 기준으로 병합 (axis=1)
merged_df = pd.concat(df_list, axis=1)

In [10]:
merged_df

,_1순위카드이용금액,이용금액_R3M_신용체크,정상청구원금_B2M,이용금액_일시불_R12M,정상입금원금_B2M,이용금액_일시불_B0M,정상청구원금_B0M,정상입금원금_B0M,정상입금원금_B5M,정상청구원금_B5M,청구금액_R3M,청구금액_B0,청구금액_R6M
0,3681,196,16524,20667,16125,1995,14440,6335,9205,14958,46588,12226,88693
1,13323,13475,2420,54341,2420,2862,6024,5198,2546,3367,10530,5834,16861
2,24493,23988,21826,55656,14448,5854,21929,12564,16949,23963,85931,21866,165221
3,5933,3904,19172,10753,13043,387,18563,7639,8418,19614,61518,16356,127371
4,0,1190,272,-2129,0,0,0,0,0,0,0,0,155
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,5640,10755,0,0,0,0,19,0,0,0,0,0,0
2399996,26357,27636,14844,148106,10764,7663,13462,9705,21831,23742,37515,14402,99849
2399997,17171,23187,6862,52233,6106,4545,7049,5346,3269,4125,22274,5731,41073
2399998,0,0,0,0,0,0,0,0,0,507,0,0,0


In [11]:
# VIF 계산 함수
def calculate_vif(df):
    df_const = add_constant(df)
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df_const.values, i + 1) for i in range(df.shape[1])]
    return vif_data

In [ ]:
# 초기값
current_df = merged_df.copy()
# current_df

In [15]:
vif_wide = pd.DataFrame()
vif_removal_log = []
columns_remaining = current_df.columns.tolist()

while current_df.shape[1] > 5:
    col_count = current_df.shape[1]

    vif_result = calculate_vif(current_df)
    vif_series = vif_result.set_index("Variable")["VIF"]
    vif_wide[f"{col_count}개"] = vif_series

    # 제거할 컬럼
    max_vif_row = vif_result.sort_values("VIF", ascending=False).iloc[0]
    vif_removal_log.append({
        '제거된_컬럼': max_vif_row['Variable'],
        '해당_VIF': max_vif_row['VIF'],
        '남은_컬럼수': col_count - 1
    })

    current_df = current_df.drop(columns=[max_vif_row['Variable']])

# 마지막 5개 남았을 때
vif_result = calculate_vif(current_df)
vif_series = vif_result.set_index("Variable")["VIF"]
vif_wide["5개"] = vif_series

# 결과 저장
vif_wide.to_csv("vif_wide_table.csv", encoding='utf-8-sig')
pd.DataFrame(vif_removal_log).to_csv("vif_removed_log.csv", index=False, encoding='utf-8-sig')

print("저장 완료: vif_wide_table.csv, vif_removed_log.csv")


저장 완료: vif_wide_table.csv, vif_removed_log.csv


In [16]:
vif_remove = pd.read_csv('vif_wide_table.csv')

In [20]:
vif_remove

,Variable,13개,12개,11개,10개,9개,8개,7개,6개,5개
0,_1순위카드이용금액,13.301579,13.131226,13.088245,12.557650,9.774795,9.414638,NaN,NaN,NaN
1,이용금액_R3M_신용체크,5.103210,5.095172,5.094633,5.091041,4.805123,4.795403,3.767963,3.732714,3.726804
2,정상청구원금_B2M,27.458484,17.212504,17.026076,NaN,NaN,NaN,NaN,NaN,NaN
3,이용금액_일시불_R12M,10.062601,10.000084,9.982335,9.861041,6.600013,6.598588,4.677615,4.407812,4.358791
4,정상입금원금_B2M,15.324693,12.566342,11.348594,7.384719,7.377911,5.645817,5.445157,5.180307,5.147520
5,이용금액_일시불_B0M,15.804099,15.769551,15.766875,15.729356,NaN,NaN,NaN,NaN,NaN
6,정상청구원금_B0M,27.899007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,정상입금원금_B0M,16.767975,12.232257,11.852101,11.700892,11.544326,NaN,NaN,NaN,NaN
8,정상입금원금_B5M,11.165146,10.758549,10.640440,7.983211,7.887877,6.530218,6.293001,4.850701,4.683646
9,정상청구원금_B5M,17.244791,15.788177,15.773842,8.452115,8.452031,8.050647,7.972470,NaN,NaN


In [23]:
nan_drop = vif_remove[['Variable', '7개']].dropna()

In [24]:
nan_drop

,Variable,7개
1,이용금액_R3M_신용체크,3.767963
3,이용금액_일시불_R12M,4.677615
4,정상입금원금_B2M,5.445157
8,정상입금원금_B5M,6.293001
9,정상청구원금_B5M,7.972470
11,청구금액_B0,5.995032
12,청구금액_R6M,7.638845


In [25]:
nan_drop_5 = vif_remove[['Variable', '5개']].dropna()
nan_drop_5

,Variable,5개
1,이용금액_R3M_신용체크,3.726804
3,이용금액_일시불_R12M,4.358791
4,정상입금원금_B2M,5.147520
8,정상입금원금_B5M,4.683646
11,청구금액_B0,3.004369


In [26]:
print(nan_drop_5)

         Variable        5개
1   이용금액_R3M_신용체크  3.726804
3   이용금액_일시불_R12M  4.358791
4      정상입금원금_B2M  5.147520
8      정상입금원금_B5M  4.683646
11        청구금액_B0  3.004369


#### 4개 선정
- 이용금액_R3M_신용체크  
- 이용금액_일시불_R12M  
- 정상입금원금_B5M  
- 청구금액_B0  

In [ ]:
from pipeline.preprocessing import merge_segment_df

selected_columns = nan_drop_5['Variable'].tolist()

paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

# 함수 실행
merged_df = merge_segment_df(selected_columns, paths)

# 결과 저장
merged_df.to_csv('./result/CD_VIF_col_5_with_segment_test.csv', encoding='utf-8-sig', index=False)

In [48]:
df = pd.read_parquet('./data/train/1_회원정보_train.parquet')

In [51]:
selected_columns = df[df['Segment'] == 'C'].columns.tolist()
selected_columns

['기준년월',
 'ID',
 '남녀구분코드',
 '연령',
 'Segment',
 '회원여부_이용가능',
 '회원여부_이용가능_CA',
 '회원여부_이용가능_카드론',
 '소지여부_신용',
 '소지카드수_유효_신용',
 '소지카드수_이용가능_신용',
 '입회일자_신용',
 '입회경과개월수_신용',
 '회원여부_연체',
 '이용거절여부_카드론',
 '동의여부_한도증액안내',
 '수신거부여부_TM',
 '수신거부여부_DM',
 '수신거부여부_메일',
 '수신거부여부_SMS',
 '가입통신회사코드',
 '탈회횟수_누적',
 '최종탈회후경과월',
 '탈회횟수_발급6개월이내',
 '탈회횟수_발급1년이내',
 '거주시도명',
 '직장시도명',
 '마케팅동의여부',
 '유효카드수_신용체크',
 '유효카드수_신용',
 '유효카드수_신용_가족',
 '유효카드수_체크',
 '유효카드수_체크_가족',
 '이용가능카드수_신용체크',
 '이용가능카드수_신용',
 '이용가능카드수_신용_가족',
 '이용가능카드수_체크',
 '이용가능카드수_체크_가족',
 '이용카드수_신용체크',
 '이용카드수_신용',
 '이용카드수_신용_가족',
 '이용카드수_체크',
 '이용카드수_체크_가족',
 '이용금액_R3M_신용체크',
 '이용금액_R3M_신용',
 '이용금액_R3M_신용_가족',
 '이용금액_R3M_체크',
 '이용금액_R3M_체크_가족',
 '_1순위카드이용금액',
 '_1순위카드이용건수',
 '_1순위신용체크구분',
 '_2순위카드이용금액',
 '_2순위카드이용건수',
 '_2순위신용체크구분',
 '최종유효년월_신용_이용가능',
 '최종유효년월_신용_이용',
 '최종카드발급일자',
 '보유여부_해외겸용_본인',
 '이용가능여부_해외겸용_본인',
 '이용여부_3M_해외겸용_본인',
 '보유여부_해외겸용_신용_본인',
 '이용가능여부_해외겸용_신용_본인',
 '이용여부_3M_해외겸용_신용_본인',
 '연회비발생카드수_B0M',
 '연회비할인카드수_B0M',
 '기본연회비_B0M',
 '제휴

In [ ]:
from pipeline.preprocessing import merge_segment_df



paths = [
    './data/train/1_회원정보_train.parquet,'
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

# 함수 실행
merged_df = merge_segment_df(selected_columns, paths)

# 결과 저장
merged_df.to_csv('./result/CD_VIF_col_5_with_segment_test.csv', encoding='utf-8-sig', index=False)

In [ ]:
# merged_df

In [47]:
import pandas as pd
from pipeline.vif_tools import stepwise_vif_table

df = pd.read_csv('./result/CD_VIF_col_5_with_segment_test.csv')

numeric_df = df.select_dtypes(include='number')

vif_df, vif_log_df = stepwise_vif_table(
    df=numeric_df,
    min_cols=4,
    output_path="./result/test1.csv",
    log_path="./result/test1_log.csv"
)

VIF 테이블 저장 완료: ./result/test1.csv
제거 로그 저장 완료: ./result/test1_log.csv
